# Candidates: SNR against the distance modulus, old and new match filter, PSF and template-fit VIS

For each candidate of `data/candidates/candidates_info.csv` (found on the maps of the old match filter with the PSF
VIS), its SNR at each distance modulus, as in the candidate investigation (`notebooks/candidate_investigation_example.ipynb`):
the stars in the match filter, outside the masks, on the on region along the candidate and in the two off regions on
its sides; SNR = excess on the on region (over the mean density of the off regions) / sqrt(background on the on
region) (`CandidateInvestigation.snr_against_distance_modulus`, `on_off_regions.selection_snr`).

Four selections: the old and the new match filter (`MATCH_FILTERS`), each with the PSF and the template-fit VIS
(`VIS_PHOTOMETRIES`). The regions of a candidate are the same for the four: centre and width from the fit of its
phi2 profile (`fit_width`) at its distance modulus (`best_dm`, else `best_dm_by_eye`) with `REFERENCE_SELECTION`, or given in `REGION_OVERRIDES`. The
template-fit VIS puts a stream about 0.2 mag farther than the PSF VIS (`generic_search_comparison.ipynb`, section 3):
expect the template-fit curves to peak about 0.2 later.

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from streameucliddes.analysis.candidate_investigation import DEFAULT_FOOTPRINT_NSIDE, CandidateInvestigation
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.catalogs.stream_candidates import load_candidates
from streameucliddes.catalogs.survey_footprint import footprint_mask_from_stars

## 1. The selections

Edit the dictionaries: a match filter is the Euclid DR1 VIS vs VIS − H filter with its width and red cap (absent: the
streamobs defaults, `error_multiplier` 2 and red cap 0.75), and its isochrone model (the default isochrone, 13 Gyr and
[Fe/H] = -1.9, its Z in the model); the VIS photometry is added to it.

In [ ]:
VIS_H = {"reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "H")]}
MATCH_FILTERS = {
    # before 2026-11-18: the streamobs width and red cap, Marigo2017
    "old": {"match_filter": {"name": "old", **VIS_H}, "isochrone_model": "Marigo2017"},
    # since: the best on Aliqa Uma (notebooks/optimization/stream_optimization.ipynb), DEFAULT_MATCH_FILTER today
    "new": {"match_filter": {"name": "new", **VIS_H, "error_multiplier": (3.0, 3.0), "red_cap_value": 0.55},
            "isochrone_model": "Dotter2016"},
}
VIS_PHOTOMETRIES = {"PSF": "psf", "template fit": "templfit"}

# Every match filter with every VIS photometry
SELECTIONS = {
    f"{filter_name} match filter, {photometry_name}": {
        "match_filter": {**definition["match_filter"], "vis_photometry": photometry},
        "isochrone_model": definition["isochrone_model"],
    }
    for (filter_name, definition), (photometry_name, photometry) in itertools.product(MATCH_FILTERS.items(),
                                                                                      VIS_PHOTOMETRIES.items())
}
# Plot styles: colour per match filter, line style per VIS photometry
FILTER_COLORS = {"old": "C0", "new": "C1"}
PHOTOMETRY_LINESTYLES = {"PSF": "-", "template fit": "--"}
PHOTOMETRY_MARKERS = {"PSF": "o", "template fit": "s"}
for name, selection in SELECTIONS.items():
    print(f"{name}: {selection}")

## 2. Candidates, stars and regions

`REFERENCE_SELECTION`: the selection of the fit of the regions, at the distance modulus of each candidate (`best_dm`, else `best_dm_by_eye`) (the old match
filter with the PSF VIS: the one the candidates were found with). `REGION_OVERRIDES`: centre and width given by hand
for a candidate (e.g. a fit gone wrong); `REGION_FALLBACK`: when the fit fails or is wider than
`MAX_FIT_SIGMA_DEG`.

In [ ]:
CATALOG = "euclid_dr1"
SUBSET = "south"  # the candidates are in the DR1 south
STAR_SELECTION = "morphology_cuts"
DISTANCE_MODULI = np.round(np.arange(15.8, 18.41, 0.1), 2)
CANDIDATE_IDS = None  # None: every candidate of the list; or e.g. [1, 2, 5]
REFERENCE_SELECTION = "old match filter, PSF"
BACKGROUND_DEGREE = 0  # of the fit of the phi2 profile: 0 constant, 1 linear
REGION_OVERRIDES = {8: {"sigma": 0.5}}  # candidate id: {"phi2_centre", "sigma"} (deg), those given replace the fit's
REGION_FALLBACK = {"phi2_centre": 0.0, "sigma": 0.3}  # deg, when the fit fails
MAX_FIT_SIGMA_DEG = 0.8  # a wider fit is taken as failed: the fallback
# phi2 half-width of the windows (deg): room for the off regions (to 6 sigma from the centre) of the widest fit
WINDOW_MIN_PHI2_HALF_WIDTH_DEG = 7.0

candidates = load_candidates().set_index("id")
candidate_ids = list(candidates.index) if CANDIDATE_IDS is None else list(CANDIDATE_IDS)
print(candidates.loc[candidate_ids])

In [ ]:
# The stars of the DR1 south, with the PSF and the template-fit VIS; one footprint for every candidate
catalog = get_euclid_stars(CATALOG, cuts=default_star_cuts(CATALOG, STAR_SELECTION), subset=SUBSET,
                           vis_photometry="templfit")
footprint = footprint_mask_from_stars(catalog["ra"].to_numpy(), catalog["dec"].to_numpy(), DEFAULT_FOOTPRINT_NSIDE)
print(f"{len(catalog):,} stars")

In [ ]:
BOX_MARGIN_DEG = 1.0  # stars of the (ra, dec) box of a window plus this margin: what prepare reads
star_ra, star_dec = catalog["ra"].to_numpy(), catalog["dec"].to_numpy()
box = {}  # the stars of the box of the current candidate: {candidate id: stars}, one candidate at a time


def box_stars(candidate_id, cand):
    """The stars of the (ra, dec) box of the window of a candidate, plus BOX_MARGIN_DEG (a small copy), selected once per
    candidate: prepare then reads them, not the whole catalog (the same window for every selection)."""
    if candidate_id not in box:
        box.clear()
        (ra_min, ra_max), (dec_min, dec_max) = cand.radec_box
        in_box = (star_dec >= dec_min - BOX_MARGIN_DEG) & (star_dec <= dec_max + BOX_MARGIN_DEG)
        if (ra_min, ra_max) != (0.0, 360.0):  # (0, 360): a window around a pole, every ra
            width = (ra_max - ra_min) % 360.0  # across ra = 0 too
            in_box &= (star_ra - ra_min + BOX_MARGIN_DEG) % 360.0 <= width + 2 * BOX_MARGIN_DEG
        box[candidate_id] = catalog[in_box]
    return box[candidate_id]


def candidate(candidate_id, selection_name):
    """The investigation of a candidate with a selection, its stars prepared (the footprint of the whole catalog)."""
    cand = CandidateInvestigation.from_candidate_id(candidate_id, distance_moduli=DISTANCE_MODULI,
                                                    min_phi2_half_width_deg=WINDOW_MIN_PHI2_HALF_WIDTH_DEG,
                                                    **SELECTIONS[selection_name])
    return cand.prepare(box_stars(candidate_id, cand), footprint_mask=footprint)


start = time.perf_counter()


def progress(message):
    """A line with the time since the start of the loops, shown at once."""
    print(f"[{time.perf_counter() - start:6.0f} s] {message}", flush=True)


# The regions of each candidate: fit with the reference selection at its distance modulus, overrides, fallback
regions = {}
for index, candidate_id in enumerate(candidate_ids, start=1):
    progress(f"regions: candidate {candidate_id} ({index}/{len(candidate_ids)})")
    reference = candidate(candidate_id, REFERENCE_SELECTION)
    try:
        reference.fit_width(background_degree=BACKGROUND_DEGREE)
        region = {"phi2_centre": reference.width_fit["phi2_centre"], "sigma": reference.width_fit["sigma"],
                  "source": "fit"}
        if region["sigma"] > MAX_FIT_SIGMA_DEG:
            region = {**REGION_FALLBACK, "source": f"fallback (fit sigma {region['sigma']:.2f} deg)"}
    except (RuntimeError, ValueError) as error:
        region = {**REGION_FALLBACK, "source": f"fallback ({error.__class__.__name__})"}
    if candidate_id in REGION_OVERRIDES:
        region = {**region, **REGION_OVERRIDES[candidate_id], "source": "override"}
    regions[candidate_id] = region
    del reference
pd.DataFrame(regions).T

## 3. SNR against the distance modulus

For each candidate and selection: `define_regions` with the regions above, then `snr_against_distance_modulus` over
`DISTANCE_MODULI`.

In [ ]:
rows = []
tasks = list(itertools.product(candidate_ids, SELECTIONS))
for index, (candidate_id, selection_name) in enumerate(tasks, start=1):
    progress(f"SNR: candidate {candidate_id}, {selection_name} ({index}/{len(tasks)})")
    cand = candidate(candidate_id, selection_name)
    region = regions[candidate_id]
    cand.define_regions(phi2_centre=region["phi2_centre"], sigma=region["sigma"])
    table = cand.snr_against_distance_modulus()
    table.insert(0, "selection", selection_name)
    table.insert(0, "candidate", candidate_id)
    rows.append(table)
    del cand
box.clear()
progress("done")
snr = pd.concat(rows, ignore_index=True)
snr.head()

In [ ]:
n_columns = 4
n_rows = int(np.ceil(len(candidate_ids) / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(4.5 * n_columns, 3.3 * n_rows), sharex=True, layout="constrained",
                         squeeze=False)
for ax, candidate_id in zip(axes.ravel(), candidate_ids):
    for (filter_name, photometry_name) in itertools.product(MATCH_FILTERS, VIS_PHOTOMETRIES):
        name = f"{filter_name} match filter, {photometry_name}"
        curve = snr[(snr["candidate"] == candidate_id) & (snr["selection"] == name)]
        ax.plot(curve["distance_modulus"], curve["snr"], color=FILTER_COLORS[filter_name],
                ls=PHOTOMETRY_LINESTYLES[photometry_name], label=name)
    ax.axvline(candidates.loc[candidate_id, "best_dm_by_eye"], color="0.6", lw=0.8)
    ax.axhline(0.0, color="0.8", lw=0.6)
    region = regions[candidate_id]
    ax.set_title(f"candidate {candidate_id}: phi2 {region['phi2_centre']:+.2f}, sigma {region['sigma']:.2f} deg",
                 fontsize="small")
for ax in axes.ravel()[len(candidate_ids):]:
    ax.set_visible(False)
for ax in axes[:, 0]:
    ax.set_ylabel("SNR")
for ax in axes[-1]:
    ax.set_xlabel("distance modulus")
axes[0, 0].legend(fontsize="x-small")
fig.suptitle("SNR of the candidates against the distance modulus (grey: best_dm_by_eye of the list)")
plt.show()

In [ ]:
# Peak SNR of each candidate and selection, and its distance modulus
peak = snr.loc[snr.groupby(["candidate", "selection"])["snr"].idxmax()]
peak.pivot(index="candidate", columns="selection", values=["snr", "distance_modulus"]).round(2)

In [ ]:
def plot_peak_per_candidate(column, ylabel, title, ax=None):
    """A column of the peak of each candidate (``peak``), one marker per selection (colour: match filter, marker: VIS
    photometry), side by side."""
    fig, ax = plt.subplots(figsize=(1.0 + 0.8 * len(candidate_ids), 4.0), layout="constrained") if ax is None \
        else (ax.figure, ax)
    positions = np.arange(len(candidate_ids))
    offsets = np.linspace(-0.25, 0.25, len(SELECTIONS))
    for offset, (filter_name, photometry_name) in zip(offsets, itertools.product(MATCH_FILTERS, VIS_PHOTOMETRIES)):
        name = f"{filter_name} match filter, {photometry_name}"
        points = peak[peak["selection"] == name].set_index("candidate").loc[candidate_ids]
        ax.plot(positions + offset, points[column], ls="none", marker=PHOTOMETRY_MARKERS[photometry_name],
                color=FILTER_COLORS[filter_name], label=name)
    ax.set(xticks=positions, xticklabels=candidate_ids, xlabel="candidate", ylabel=ylabel, title=title)
    ax.grid(axis="y", color="0.9", lw=0.5)
    return fig, ax


# Peak SNR of each candidate
fig, ax = plot_peak_per_candidate(
    "snr", "peak SNR", f"Peak SNR of the candidates over dm {DISTANCE_MODULI[0]:g} to {DISTANCE_MODULI[-1]:g}")
ax.axhline(0.0, color="0.8", lw=0.6)
ax.legend(fontsize="small")
plt.show()

In [ ]:
# Distance modulus of the peak of each candidate: the template-fit VIS about 0.2 later than the PSF VIS? Grey: the
# best_dm_by_eye of the list (found with the old match filter and the PSF VIS)
fig, ax = plot_peak_per_candidate("distance_modulus", "distance modulus of the peak SNR",
                                  "Distance modulus of the peak SNR of the candidates")
ax.plot(np.arange(len(candidate_ids)), candidates.loc[candidate_ids, "best_dm_by_eye"], ls="none", marker="_", ms=14,
        color="0.5", label="best_dm_by_eye of the list")
ax.legend(fontsize="small")
plt.show()